In [1]:
# Local-only setup cell
# Assumes the required packages are already installed in your active notebook kernel.
from pathlib import Path

DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = DATA_DIR / 'Submission files codex/submission.csv'

print('Running in local mode')
print('DATA_DIR =', DATA_DIR)
print('TRAIN_PATH =', TRAIN_PATH)
print('TEST_PATH =', TEST_PATH)

if not (TRAIN_PATH.exists() and TEST_PATH.exists()):
    print('\nWarning: train/test files not found in the current working directory.')
    print('Update DATA_DIR if your files are stored elsewhere.')

Running in local mode
DATA_DIR = .
TRAIN_PATH = train.csv
TEST_PATH = test.csv


In [2]:
import random
import time
import warnings
from pathlib import Path

import lightgbm as lgb
import numpy as np
import pandas as pd
import pygeohash as pgh
import torch
import torch.nn as nn
from sklearn.metrics import r2_score

warnings.filterwarnings('ignore')

RANDOM_STATE = 42
LGB_DEVICE_TYPE = 'gpu'  # set to 'cpu' if GPU LightGBM is unavailable
DATA_DIR = Path('.')
TRAIN_PATH = DATA_DIR / 'train.csv'
TEST_PATH = DATA_DIR / 'test.csv'
SUBMISSION_PATH = DATA_DIR / 'Submission files codex' / 'submission.csv'

MODEL_TYPE = 'hybrid'  # 'lgbm', 'lstm', or 'hybrid'

LAG_FEATURES = [1, 2, 4, 8, 12, 24, 96]
ROLLING_WINDOWS = [4, 8, 24]
DYNAMIC_FEATURE_COLUMNS = [
    *[f'demand_lag_{lag}' for lag in LAG_FEATURES],
    *[f'demand_roll_mean_{window}' for window in ROLLING_WINDOWS],
    *[f'demand_roll_std_{window}' for window in ROLLING_WINDOWS],
]


def set_global_seed(seed: int = 42) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


def parse_timestamp_to_minutes(timestamp_series):
    '''Convert HH:MM strings into minutes past midnight.'''
    parts = timestamp_series.astype(str).str.split(':', n=1, expand=True)
    hours = parts[0].astype(int)
    minutes = parts[1].astype(int)
    return hours * 60 + minutes


def decode_single_geohash(geohash_value):
    '''Decode one geohash string into latitude and longitude.'''
    if pd.isna(geohash_value):
        return np.nan, np.nan

    try:
        decoded = pgh.decode_exactly(str(geohash_value))
        if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
            return float(decoded.latitude), float(decoded.longitude)
        return float(decoded[0]), float(decoded[1])
    except Exception:
        try:
            decoded = pgh.decode(str(geohash_value))
            if hasattr(decoded, 'latitude') and hasattr(decoded, 'longitude'):
                return float(decoded.latitude), float(decoded.longitude)
            return float(decoded[0]), float(decoded[1])
        except Exception:
            return np.nan, np.nan


def add_geohash_coordinates(frame):
    '''Add latitude and longitude columns derived from geohash values.'''
    geohash_series = frame['geohash'].astype('string')
    unique_geohashes = geohash_series.dropna().unique()
    geohash_lookup = {value: decode_single_geohash(value) for value in unique_geohashes}

    coordinates = geohash_series.map(geohash_lookup)
    frame['latitude'] = [pair[0] if isinstance(pair, tuple) else np.nan for pair in coordinates]
    frame['longitude'] = [pair[1] if isinstance(pair, tuple) else np.nan for pair in coordinates]
    return frame


def add_lag_and_rolling_features(frame):
    '''Add autoregressive lags and rolling demand statistics by geohash.'''
    grouped = frame.groupby('geohash', sort=False)

    for lag in LAG_FEATURES:
        frame[f'demand_lag_{lag}'] = grouped['demand'].shift(lag)

    lagged_demand = grouped['demand'].shift(1)
    grouped_lagged = lagged_demand.groupby(frame['geohash'], sort=False)

    for window in ROLLING_WINDOWS:
        frame[f'demand_roll_mean_{window}'] = grouped_lagged.transform(
            lambda series: series.rolling(window=window, min_periods=1).mean()
        )
        frame[f'demand_roll_std_{window}'] = grouped_lagged.transform(
            lambda series: series.rolling(window=window, min_periods=1).std()
        )

    geohash_demand_mean = frame.groupby('geohash')['demand'].transform('mean')
    frame[DYNAMIC_FEATURE_COLUMNS] = frame[DYNAMIC_FEATURE_COLUMNS].fillna(geohash_demand_mean)
    return frame


def compute_dynamic_features(history_values):
    '''Build lag and rolling features from the available demand history.'''
    feature_values = {}

    for lag in LAG_FEATURES:
        feature_values[f'demand_lag_{lag}'] = history_values[-lag] if len(history_values) >= lag else np.nan

    for window in ROLLING_WINDOWS:
        window_values = history_values[-window:]
        if window_values:
            feature_values[f'demand_roll_mean_{window}'] = float(np.mean(window_values))
            feature_values[f'demand_roll_std_{window}'] = float(np.std(window_values, ddof=1)) if len(window_values) > 1 else 0.0
        else:
            feature_values[f'demand_roll_mean_{window}'] = np.nan
            feature_values[f'demand_roll_std_{window}'] = np.nan

    return feature_values


def recursive_predict(model, history_df, target_df, feature_columns, categorical_columns, category_levels):
    '''Predict rows sequentially while updating lag features with prior predictions.'''
    if target_df.empty:
        return pd.Series(dtype=float)

    static_columns = [column for column in feature_columns if column not in DYNAMIC_FEATURE_COLUMNS]
    predictions = []
    prediction_index = []

    history_by_geohash = {
        str(geohash_value): group.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
        for geohash_value, group in history_df.groupby('geohash', sort=False)
    }

    sorted_target_df = target_df.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort')
    for geohash_value, geohash_group in sorted_target_df.groupby('geohash', sort=False):
        geohash_key = str(geohash_value)
        history_values = history_by_geohash.get(geohash_key)
        if history_values is None:
            history_values = []
            history_by_geohash[geohash_key] = history_values

        for row in geohash_group.itertuples(index=False):
            feature_row = {column: getattr(row, column) for column in static_columns}
            feature_row.update(compute_dynamic_features(history_values))
            feature_frame = pd.DataFrame([feature_row], columns=feature_columns)

            for column in categorical_columns:
                feature_frame[column] = pd.Categorical(feature_frame[column], categories=category_levels[column])

            prediction_values = np.asarray(model.predict(feature_frame), dtype=float)
            prediction = float(prediction_values[0])
            predictions.append(prediction)
            prediction_index.append(str(getattr(row, 'Index')))
            history_values.append(prediction)

    return pd.Series(predictions, index=prediction_index)


def align_predictions(pred_series, index_list):
    mapping = {int(k): float(v) for k, v in pred_series.to_dict().items()}
    preds = np.array([mapping.get(int(idx), np.nan) for idx in index_list], dtype=float)
    if np.isnan(preds).any():
        med = np.nanmedian(preds)
        if np.isnan(med):
            med = 0.0
        preds = np.where(np.isnan(preds), med, preds)
    return preds


set_global_seed(RANDOM_STATE)

In [3]:
train_raw = pd.read_csv(TRAIN_PATH)
test_raw = pd.read_csv(TEST_PATH)

train_raw['is_test'] = False
test_raw['is_test'] = True
test_raw['demand'] = np.nan

combined_df = pd.concat([train_raw, test_raw], ignore_index=True, sort=False)
combined_df['time_mins'] = parse_timestamp_to_minutes(combined_df['timestamp']).astype(int)
combined_df['time_sin'] = np.sin(2 * np.pi * combined_df['time_mins'] / 1440.0)
combined_df['time_cos'] = np.cos(2 * np.pi * combined_df['time_mins'] / 1440.0)
combined_df = add_geohash_coordinates(combined_df)

combined_df.head()

,Index,geohash,day,timestamp,demand,RoadType,NumberofLanes,LargeVehicles,Landmarks,Temperature,Weather,is_test,time_mins,time_sin,time_cos,latitude,longitude
0,0,qp02z1,48,0:0,0.048804,NaN,1,Not Allowed,No,NaN,NaN,False,0,0.0,1.0,-5.484924,90.664673
1,1,qp02zt,48,0:0,0.118507,Residential,3,Allowed,Yes,31.104565,Sunny,False,0,0.0,1.0,-5.462952,90.686646
2,2,qp08bj,48,0:0,0.027132,Residential,1,Not Allowed,No,25.919267,Sunny,False,0,0.0,1.0,-5.462952,90.708618
3,3,qp08gt,48,0:0,0.003272,Residential,1,Not Allowed,No,NaN,Rainy,False,0,0.0,1.0,-5.462952,90.862427
4,4,qp02zq,48,0:0,0.010819,Residential,1,Not Allowed,No,10.803667,Rainy,False,0,0.0,1.0,-5.457458,90.675659


In [4]:
combined_df = combined_df.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort').reset_index(drop=True)

combined_df['Temperature'] = combined_df.groupby('geohash')['Temperature'].transform(lambda series: series.ffill().bfill())
combined_df['Weather'] = combined_df.groupby('geohash')['Weather'].transform(lambda series: series.ffill().bfill())
combined_df['RoadType'] = combined_df['RoadType'].fillna('Unknown')

combined_df = add_lag_and_rolling_features(combined_df)

categorical_columns = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']
for column_name in categorical_columns:
    combined_df[column_name] = combined_df[column_name].astype('category')

combined_df.dtypes

Index                     int64
geohash                category
day                       int64
timestamp                object
demand                  float64
RoadType               category
NumberofLanes             int64
LargeVehicles          category
Landmarks              category
Temperature             float64
Weather                category
is_test                    bool
time_mins                 int64
time_sin                float64
time_cos                float64
latitude                float64
longitude               float64
demand_lag_1            float64
demand_lag_2            float64
demand_lag_4            float64
demand_lag_8            float64
demand_lag_12           float64
demand_lag_24           float64
demand_lag_96           float64
demand_roll_mean_4      float64
demand_roll_std_4       float64
demand_roll_mean_8      float64
demand_roll_std_8       float64
demand_roll_mean_24     float64
demand_roll_std_24      float64
dtype: object

In [5]:
train_df = combined_df.loc[~combined_df['is_test']].copy()
test_df = combined_df.loc[combined_df['is_test']].copy()

feature_columns = [
    'geohash',
    'day',
    'time_mins',
    'time_sin',
    'time_cos',
    'latitude',
    'longitude',
    'RoadType',
    'NumberofLanes',
    'LargeVehicles',
    'Landmarks',
    'Temperature',
    'Weather',
    'demand_lag_1',
    'demand_lag_2',
    'demand_lag_4',
    'demand_lag_8',
    'demand_lag_12',
    'demand_lag_24',
    'demand_lag_96',
    'demand_roll_mean_4',
    'demand_roll_mean_8',
    'demand_roll_mean_24',
    'demand_roll_std_4',
    'demand_roll_std_8',
    'demand_roll_std_24',
]

train_mask = (train_df['day'] == 48)
valid_mask = (train_df['day'] == 49) & (train_df['time_mins'] <= 120)

X_train = train_df.loc[train_mask, feature_columns].copy()
y_train = train_df.loc[train_mask, 'demand'].astype(float)
X_valid = train_df.loc[valid_mask, feature_columns].copy()
y_valid = train_df.loc[valid_mask, 'demand'].astype(float)


def build_regressor(device_type: str, n_estimators: int) -> lgb.LGBMRegressor:
    '''Create a LightGBM regressor with the requested accelerator mode.'''
    return lgb.LGBMRegressor(
        objective='regression',
        random_state=RANDOM_STATE,
        n_estimators=n_estimators,
        learning_rate=0.05,
        device_type=device_type,
        n_jobs=-1,
        verbosity=-1,
        max_depth=-1,
        max_bin=255,
        num_leaves=255,
        min_child_samples=20,
        subsample=0.9,
        subsample_freq=1,
        colsample_bytree=0.9,
        reg_alpha=0.1,
        reg_lambda=0.2,
    )


try:
    model = build_regressor(LGB_DEVICE_TYPE, n_estimators=1500)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=True)],
    )
    trained_device = LGB_DEVICE_TYPE
except lgb.basic.LightGBMError as error:
    # if LGB_DEVICE_TYPE != 'gpu' or ('CUDA Tree Learner was not enabled' not in str(error) and 'GPU Tree Learner was not enabled' not in str(error)):
    #     raise
    print('GPU build is unavailable in this environment. Falling back to CPU training.')
    model = build_regressor('cpu', 1500)
    model.fit(
        X_train,
        y_train,
        eval_set=[(X_valid, y_valid)],
        eval_metric='rmse',
        categorical_feature=categorical_columns,
        callbacks=[lgb.early_stopping(50, verbose=True)],
    )
    trained_device = 'cpu'

category_levels = {column: combined_df[column].cat.categories for column in categorical_columns}
validation_history_df = train_df.loc[train_df['day'] == 48].copy()
valid_target_df = train_df.loc[valid_mask].copy()
valid_prediction_series = recursive_predict(
    model,
    validation_history_df,
    valid_target_df,
    feature_columns,
    categorical_columns,
    category_levels,
)
# Use align_predictions to robustly map the recursive-predict series to ordered numpy array
valid_predictions = align_predictions(valid_prediction_series, valid_target_df['Index'].astype(int).tolist())
valid_r2 = r2_score(y_valid, valid_predictions)
print(f'Validation R2: {valid_r2:.6f}')
print(f'Validation Score: {max(0.0, 100 * valid_r2):.4f}')
print(f'Training device used: {trained_device}')

best_n_estimators = model.best_iteration_ or 1500
full_train_df = train_df.loc[train_df['demand'].notna()].copy()
X_full_train = full_train_df[feature_columns].copy()
y_full_train = full_train_df['demand'].astype(float)

final_model = build_regressor(trained_device, best_n_estimators)
final_model.fit(
    X_full_train,
    y_full_train,
    categorical_feature=categorical_columns,
)

test_history_df = train_df.copy()
test_prediction_series = recursive_predict(
    final_model,
    test_history_df,
    test_df.copy(),
    feature_columns,
    categorical_columns,
    category_levels,
)
# Use align_predictions to map to the original test Index ordering
test_predictions = align_predictions(test_prediction_series, test_df['Index'].astype(int).tolist())
submission = pd.DataFrame({
    'Index': test_df['Index'].astype(int),
    'demand': test_predictions,
})
submission.to_csv(SUBMISSION_PATH, index=False)
submission.head()

GPU build is unavailable in this environment. Falling back to CPU training.
Training until validation scores don't improve for 50 rounds
Early stopping, best iteration is:
[111]	valid_0's rmse: 0.0296812	valid_0's l2: 0.000880972
Validation R2: 0.915908
Validation Score: 91.5908
Training device used: cpu


,Index,demand
12,887,0.024478
13,4491,0.025401
14,21800,0.021762
15,40211,0.019998
17,2,0.027892


In [6]:
# LSTM / Hybrid model dispatcher (no package installs here)
from torch.utils.data import DataLoader, TensorDataset

SEQ_LEN = 24
LSTM_HIDDEN = 64
LSTM_EPOCHS = 20
LSTM_BATCH_SIZE = 1024
LSTM_LR = 1e-3
LSTM_PATIENCE = 4

DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('MODEL_TYPE =', MODEL_TYPE)
print('Torch device =', DEVICE)

lstm_numeric_columns = [
    'day',
    'time_mins',
    'time_sin',
    'time_cos',
    'latitude',
    'longitude',
    'NumberofLanes',
    'Temperature',
]
lstm_cat_columns = ['geohash', 'RoadType', 'Weather', 'LargeVehicles', 'Landmarks']

lstm_frame = combined_df.copy().sort_values(['geohash', 'day', 'time_mins'], kind='mergesort').reset_index(drop=True)
for col in lstm_cat_columns:
    lstm_frame[f'{col}_code'] = lstm_frame[col].cat.codes.astype('float32')

for col in lstm_numeric_columns:
    lstm_frame[col] = lstm_frame[col].astype('float32')

lstm_context_columns = lstm_numeric_columns + [f'{col}_code' for col in lstm_cat_columns]

geo_train_mean = train_df.groupby('geohash', observed=False)['demand'].mean().to_dict()
global_mean = float(train_df['demand'].mean())


def _get_geo_fill(geohash_value: str) -> float:
    return float(geo_train_mean.get(geohash_value, global_mean))


def _context_vector_from_row(row_obj) -> np.ndarray:
    vals = [float(getattr(row_obj, col)) for col in lstm_context_columns]
    return np.asarray(vals, dtype=np.float32)


def _build_train_windows(max_day: int = 48) -> tuple[np.ndarray, np.ndarray, np.ndarray]:
    seq_list: list[np.ndarray] = []
    ctx_list: list[np.ndarray] = []
    y_list: list[float] = []

    train_part = lstm_frame.loc[(~lstm_frame['is_test']) & (lstm_frame['day'] <= max_day)].copy()
    for geohash_value, group in train_part.groupby('geohash', sort=False):
        group = group.sort_values(['day', 'time_mins'])
        history: list[float] = []
        fill_val = _get_geo_fill(str(geohash_value))

        for row in group.itertuples(index=False):
            target = float(row.demand)
            seq = history[-SEQ_LEN:]
            if len(seq) < SEQ_LEN:
                seq = [fill_val] * (SEQ_LEN - len(seq)) + seq

            seq_arr = np.asarray(seq, dtype=np.float32).reshape(SEQ_LEN, 1)
            ctx_arr = _context_vector_from_row(row)

            seq_list.append(seq_arr)
            ctx_list.append(ctx_arr)
            y_list.append(target)

            history.append(target)

    return np.stack(seq_list), np.stack(ctx_list), np.asarray(y_list, dtype=np.float32)


class DemandLSTM(nn.Module):
    def __init__(self, context_dim: int, hidden_dim: int = 64):
        super().__init__()
        self.lstm = nn.LSTM(input_size=1, hidden_size=hidden_dim, num_layers=1, batch_first=True)
        self.head = nn.Sequential(
            nn.Linear(hidden_dim + context_dim, 128),
            nn.ReLU(),
            nn.Dropout(0.15),
            nn.Linear(128, 1),
        )

    def forward(self, seq_x: torch.Tensor, ctx_x: torch.Tensor) -> torch.Tensor:
        lstm_out, _ = self.lstm(seq_x)
        last_hidden = lstm_out[:, -1, :]
        fused = torch.cat([last_hidden, ctx_x], dim=1)
        pred = self.head(fused)
        return pred.squeeze(1)


def train_lstm_model() -> DemandLSTM:
    X_seq, X_ctx, y = _build_train_windows(max_day=48)

    n_total = len(y)
    n_val = max(2048, int(0.1 * n_total))
    n_val = min(n_val, n_total // 3)
    split = n_total - n_val

    tr_seq, va_seq = X_seq[:split], X_seq[split:]
    tr_ctx, va_ctx = X_ctx[:split], X_ctx[split:]
    tr_y, va_y = y[:split], y[split:]

    train_ds = TensorDataset(
        torch.from_numpy(tr_seq),
        torch.from_numpy(tr_ctx),
        torch.from_numpy(tr_y),
    )
    val_ds = TensorDataset(
        torch.from_numpy(va_seq),
        torch.from_numpy(va_ctx),
        torch.from_numpy(va_y),
    )

    train_loader = DataLoader(train_ds, batch_size=LSTM_BATCH_SIZE, shuffle=True, drop_last=False)
    val_loader = DataLoader(val_ds, batch_size=LSTM_BATCH_SIZE, shuffle=False, drop_last=False)

    model_lstm = DemandLSTM(context_dim=X_ctx.shape[1], hidden_dim=LSTM_HIDDEN).to(DEVICE)
    optimizer = torch.optim.Adam(model_lstm.parameters(), lr=LSTM_LR)
    criterion = nn.MSELoss()

    best_state = None
    best_val_loss = float('inf')
    bad_epochs = 0

    for epoch in range(1, LSTM_EPOCHS + 1):
        t0 = time.time()
        model_lstm.train()
        train_losses = []

        for b_seq, b_ctx, b_y in train_loader:
            b_seq = b_seq.to(DEVICE)
            b_ctx = b_ctx.to(DEVICE)
            b_y = b_y.to(DEVICE)

            optimizer.zero_grad(set_to_none=True)
            pred = model_lstm(b_seq, b_ctx)
            loss = criterion(pred, b_y)
            loss.backward()
            optimizer.step()
            train_losses.append(float(loss.item()))

        model_lstm.eval()
        val_losses = []
        with torch.no_grad():
            for b_seq, b_ctx, b_y in val_loader:
                b_seq = b_seq.to(DEVICE)
                b_ctx = b_ctx.to(DEVICE)
                b_y = b_y.to(DEVICE)
                pred = model_lstm(b_seq, b_ctx)
                val_losses.append(float(criterion(pred, b_y).item()))

        tr_loss = float(np.mean(train_losses)) if train_losses else np.nan
        va_loss = float(np.mean(val_losses)) if val_losses else np.nan
        elapsed = time.time() - t0
        print(f'Epoch {epoch:02d}: train_loss={tr_loss:.6f}, val_loss={va_loss:.6f}, sec={elapsed:.1f}')

        if va_loss < best_val_loss:
            best_val_loss = va_loss
            best_state = {k: v.detach().cpu().clone() for k, v in model_lstm.state_dict().items()}
            bad_epochs = 0
        else:
            bad_epochs += 1
            if bad_epochs >= LSTM_PATIENCE:
                print(f'Early stopping at epoch {epoch}')
                break

    if best_state is not None:
        model_lstm.load_state_dict(best_state)
    return model_lstm


def recursive_lstm_predict(
    model_lstm: DemandLSTM,
    history_df_local: pd.DataFrame,
    target_df_local: pd.DataFrame,
) -> pd.Series:
    if target_df_local.empty:
        return pd.Series(dtype=float)

    model_lstm.eval()

    history_by_geo: dict[str, list[float]] = {
        str(gh): grp.sort_values(['day', 'time_mins'])['demand'].astype(float).tolist()
        for gh, grp in history_df_local.groupby('geohash', sort=False)
    }

    target_sorted = target_df_local.sort_values(['geohash', 'day', 'time_mins'], kind='mergesort')
    preds: list[float] = []
    pred_idx: list[int] = []

    with torch.no_grad():
        for gh, grp in target_sorted.groupby('geohash', sort=False):
            gh_key = str(gh)
            fill_val = _get_geo_fill(gh_key)
            hist = history_by_geo.get(gh_key, [])

            for row in grp.itertuples(index=False):
                seq = hist[-SEQ_LEN:]
                if len(seq) < SEQ_LEN:
                    seq = [fill_val] * (SEQ_LEN - len(seq)) + seq

                seq_tensor = torch.from_numpy(np.asarray(seq, dtype=np.float32).reshape(1, SEQ_LEN, 1)).to(DEVICE)
                ctx_tensor = torch.from_numpy(_context_vector_from_row(row).reshape(1, -1)).to(DEVICE)

                p = float(model_lstm(seq_tensor, ctx_tensor).item())
                preds.append(p)
                pred_idx.append(int(getattr(row, 'Index')))
                hist.append(p)

    return pd.Series(preds, index=pred_idx)


def prepare_lstm_df(df: pd.DataFrame) -> pd.DataFrame:
    """Applies the same categorical encoding and type casting used on lstm_frame."""
    df_copy = df.copy()
    for col in lstm_cat_columns:
        df_copy[f'{col}_code'] = df_copy[col].cat.codes.astype('float32')
    for col in lstm_numeric_columns:
        df_copy[col] = df_copy[col].astype('float32')
    return df_copy

if MODEL_TYPE == 'lgbm':
    output_df = pd.DataFrame({'Index': test_df['Index'].astype(int), 'demand': test_predictions})
    output_name = 'submission_lgbm.csv'

else:
    t_train = time.time()
    lstm_model = train_lstm_model()
    print(f'LSTM train time: {time.time() - t_train:.1f}s')

    # FIX: Wrap prediction inputs with prepare_lstm_df()
    lstm_val_series = recursive_lstm_predict(
        lstm_model,
        prepare_lstm_df(train_df.loc[train_df['day'] <= 48]),
        prepare_lstm_df(valid_target_df),
    )
    lstm_valid = align_predictions(lstm_val_series, valid_target_df['Index'].astype(int).tolist())
    lstm_r2 = r2_score(y_valid, lstm_valid)
    print(f'LSTM recursive validation R2: {lstm_r2:.6f}')

    # FIX: Wrap prediction inputs with prepare_lstm_df()
    lstm_test_series = recursive_lstm_predict(
        lstm_model,
        prepare_lstm_df(train_df),
        prepare_lstm_df(test_df),
    )
    lstm_test = align_predictions(lstm_test_series, test_df['Index'].astype(int).tolist())

    if MODEL_TYPE == 'hybrid':
        if 'valid_predictions' not in globals() or 'test_predictions' not in globals():
            raise RuntimeError('Run Cell 7 first so LightGBM validation/test predictions are available for hybrid mode.')

        best_alpha = 0.0
        best_r2 = -1e9
        for alpha in np.linspace(0.0, 1.0, 11):
            blend_valid = alpha * lstm_valid + (1.0 - alpha) * valid_predictions
            score = r2_score(y_valid, blend_valid)
            if score > best_r2:
                best_r2 = score
                best_alpha = float(alpha)

        blend_test = best_alpha * lstm_test + (1.0 - best_alpha) * test_predictions
        print(f'Hybrid best alpha={best_alpha:.2f}, validation R2={best_r2:.6f}')
        output_df = pd.DataFrame({'Index': test_df['Index'].astype(int), 'demand': blend_test})
        output_name = 'submission_hybrid.csv'

    else:
        output_df = pd.DataFrame({'Index': test_df['Index'].astype(int), 'demand': lstm_test})
        output_name = 'submission_lstm.csv'

output_df.to_csv(SUBMISSION_PATH.with_name(output_name), index=False)
print('Wrote', output_name)
output_df.head()

MODEL_TYPE = hybrid
Torch device = cuda
Epoch 01: train_loss=118.848621, val_loss=0.448053, sec=0.9
Epoch 02: train_loss=42.010407, val_loss=0.994924, sec=0.7
Epoch 03: train_loss=21.848157, val_loss=0.588136, sec=0.6
Epoch 04: train_loss=11.710899, val_loss=0.604355, sec=0.6
Epoch 05: train_loss=6.975299, val_loss=0.167932, sec=0.6
Epoch 06: train_loss=4.447270, val_loss=0.136704, sec=0.6
Epoch 07: train_loss=2.966231, val_loss=0.127860, sec=0.6
Epoch 08: train_loss=2.104278, val_loss=0.091514, sec=0.6
Epoch 09: train_loss=1.501683, val_loss=0.097737, sec=0.6
Epoch 10: train_loss=1.110019, val_loss=0.086876, sec=0.7
Epoch 11: train_loss=0.861939, val_loss=0.055330, sec=0.6
Epoch 12: train_loss=0.658237, val_loss=0.050705, sec=0.6
Epoch 13: train_loss=0.531318, val_loss=0.061242, sec=0.6
Epoch 14: train_loss=0.423871, val_loss=0.038231, sec=0.6
Epoch 15: train_loss=0.358829, val_loss=0.071284, sec=0.6
Epoch 16: train_loss=0.285291, val_loss=0.035221, sec=0.6
Epoch 17: train_loss=0.2390

,Index,demand
12,887,0.024478
13,4491,0.025401
14,21800,0.021762
15,40211,0.019998
17,2,0.027892


In [11]:
# --- LOCAL EVALUATION SCRIPT ---
import os


if os.path.exists(DATA_DIR):
    print("\nEvaluating against ground truth...")
    actual_df = pd.read_csv('submission.csv').sort_values('Index').reset_index(drop=True)
    pred_df = pd.read_csv(DATA_DIR / 'Submission files codex' / 'submission_hybrid.csv').sort_values('Index').reset_index(drop=True)
    
    r2 = r2_score(actual_df['demand'], pred_df['demand'])
    print("="*45)
    print(" 🚀 PYTORCH Bi-LSTM EVALUATION 🚀")
    print("="*45)
    print(f"True R2 Score: {r2:.6f}")
    print(f"Leaderboard Score: {max(0, 100 * r2):.4f} / 100")
    print("="*45)


Evaluating against ground truth...
 🚀 PYTORCH Bi-LSTM EVALUATION 🚀
True R2 Score: 0.899990
Leaderboard Score: 89.9990 / 100
